# Hybrid LSTM+GAT Fraud Detection — IBM & Sparkov
Standalone companion to *Isolating Graph Topology from Model Architecture in
GNN-Based Fraud Detection* (Amiri & Jaf, University of Sunderland, 2026).
Extends the fixed-architecture, variable-topology framework with three
architecture variants per dataset, holding topology fixed and varying
architecture instead — the inverse experiment.

| File | Architecture | Node granularity |
|---|---|---|
| `<dataset>/lstm_gat_sequential_model.py` | LSTM → GAT pipeline | Transaction |
| `<dataset>/lstm_gat_parallel_model.py` | LSTM ‖ GAT, cross-attention fusion | Transaction |
| `<dataset>/account_gat_homogeneous_model.py` | Unchanged GATv2, new topology | Account |

Run the **IBM** section, the **Sparkov** section, or both — they're
independent and don't share state.

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Update `REPO_URL` in Cell 2
3. Skip whichever dataset section you don't need

---
## Cell 1 — Install dependencies

In [1]:
import subprocess, sys
import torch

torch_version = torch.__version__.split('+')[0]
cuda_version  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_version}  |  CUDA: {cuda_version}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'pandas', 'matplotlib', 'pyarrow', 'faiss-cpu'], check=True)
print('Done.')

PyTorch 2.11.0  |  CUDA: cu128
Done.


---
## Cell 2 — Clone this repo

In [2]:
import os

REPO_URL = 'https://github.com/Roya62/hybrid-gnn-lstm-fraud'  # ← update

if not os.path.isdir('/content/hybrid-gnn-lstm-fraud'):
    !git clone -q {REPO_URL} /content/hybrid-gnn-lstm-fraud

os.makedirs('/content/outcomes', exist_ok=True)
print('Repo ready at /content/hybrid-gnn-lstm-fraud')

Repo ready at /content/hybrid-gnn-lstm-fraud


---
## Cell 2b — (Optional) Mount Google Drive
Only needed if your dataset files live in Drive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
# Part A — IBM Dataset

## Cell 3 — Load and preprocess
Set `IBM_DATA_PATH` to your actual `reduced_dataset.parquet` (or
equivalent) location before running.

In [4]:
import os, sys
import sys

for mod in ['config', 'utils', 'gatv2_model', 'lstm_gat_sequential_model',
            'lstm_gat_parallel_model', 'account_gat_homogeneous_model']:
    sys.modules.pop(mod, None)

import os, sys

IBM_DATA_PATH = '/content/drive/MyDrive/reduced_dataset.parquet'  # ← update

os.chdir('/content/hybrid-gnn-lstm-fraud/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes/ibm'

import utils as ibm_utils
df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)
print(f'{len(df_ibm):,} transactions loaded.')

Loaded: 24,386,834 transactions
Downsampled: 29,757 fraud + 297,570 non-fraud
After preprocessing: 327,327 rows | Fraud rate: 0.0909
327,327 transactions loaded.


---
## Cell 4 — IBM: GATv2 baseline (fixed architecture, for comparison)
~25–40 min on T4 GPU (all 3 strategies).

In [5]:
import gatv2_model as ibm_gatv2

gatv2_results_ibm = ibm_gatv2.run_all_strategies(df_ibm, cfg_ibm)
print('\nIBM GATv2 baseline done.')


############################################################
# GATv2 | multi_relation
############################################################
  Fold 1/5
  [multi_relation] nodes=211,649 edges=916,245 deg(min/med/mean/max)=(1,5.0,4.3,9)
  [multi_relation] nodes=51,829 edges=267,895 deg(min/med/mean/max)=(1,5.0,5.2,9)
     TRAIN | F1 0.823 | P 0.837 | R 0.808 | AUC 0.981 | AP 0.888 | LL 0.2263 | Brier 0.0666
       VAL | F1 0.808 | P 0.833 | R 0.784 | AUC 0.977 | AP 0.874 | LL 0.2416 | Brier 0.0715
  Fold 2/5
     TRAIN | F1 0.831 | P 0.844 | R 0.819 | AUC 0.984 | AP 0.905 | LL 0.2188 | Brier 0.0640
       VAL | F1 0.805 | P 0.826 | R 0.784 | AUC 0.978 | AP 0.883 | LL 0.2509 | Brier 0.0748
  Fold 3/5
     TRAIN | F1 0.824 | P 0.841 | R 0.809 | AUC 0.982 | AP 0.894 | LL 0.2241 | Brier 0.0652
       VAL | F1 0.822 | P 0.842 | R 0.803 | AUC 0.982 | AP 0.893 | LL 0.2395 | Brier 0.0712
  Fold 4/5
     TRAIN | F1 0.829 | P 0.852 | R 0.808 | AUC 0.984 | AP 0.900 | LL 0.2115 | Brier 0.0624

---
## Cell 5 — IBM: Sequential LSTM→GAT
~35–55 min on T4 GPU.

In [6]:
import lstm_gat_sequential_model as ibm_lstm_seq
import pickle, os

lstm_seq_results_ibm = {}
for strat in ["multi_relation",
              "hybrid", "intra_group"
              ]:
    save_path = f'/content/drive/MyDrive/lstm_seq_{strat}_result.pkl'
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            lstm_seq_results_ibm[f"{strat}_lstm_gat_seq"] = {'test_metrics': pickle.load(f)}
        print(f"Skipping {strat} — already saved.")
        continue
    lstm_seq_results_ibm[f"{strat}_lstm_gat_seq"] = ibm_lstm_seq.run_pipeline(df_ibm, cfg_ibm, strat)
    with open(save_path, 'wb') as f:
        pickle.dump(lstm_seq_results_ibm[f"{strat}_lstm_gat_seq"]['test_metrics'], f)
    print(f"Saved {strat} to Drive.")

print('\nIBM Sequential LSTM→GAT done.')


############################################################
# LSTM->GAT | multi_relation
############################################################
  Fold 1/5
  [multi_relation] nodes=211,649 edges=916,245 deg(min/med/mean/max)=(1,5.0,4.3,9)
  [multi_relation] nodes=51,829 edges=267,895 deg(min/med/mean/max)=(1,5.0,5.2,9)
     TRAIN | F1 0.889 | P 0.887 | R 0.890 | AUC 0.993 | AP 0.952 | LL 0.1129 | Brier 0.0312
       VAL | F1 0.880 | P 0.901 | R 0.859 | AUC 0.990 | AP 0.941 | LL 0.1078 | Brier 0.0297
  Fold 2/5
     TRAIN | F1 0.894 | P 0.916 | R 0.873 | AUC 0.994 | AP 0.955 | LL 0.1087 | Brier 0.0304
       VAL | F1 0.876 | P 0.902 | R 0.851 | AUC 0.990 | AP 0.942 | LL 0.1222 | Brier 0.0348
  Fold 3/5
     TRAIN | F1 0.882 | P 0.883 | R 0.882 | AUC 0.993 | AP 0.948 | LL 0.1344 | Brier 0.0374
       VAL | F1 0.876 | P 0.875 | R 0.877 | AUC 0.990 | AP 0.939 | LL 0.1455 | Brier 0.0412
  Fold 4/5
     TRAIN | F1 0.897 | P 0.911 | R 0.882 | AUC 0.994 | AP 0.957 | LL 0.1199 | Brier 0.

---
## Cell 6 — IBM: Parallel LSTM‖GAT
~45–70 min on T4 GPU.

In [7]:
import lstm_gat_parallel_model as ibm_lstm_par
import pickle, os

lstm_par_results_ibm = {}
for strat in ["multi_relation",
              "hybrid", "intra_group"]:
    save_path = f'/content/drive/MyDrive/lstm_par_{strat}_result.pkl'
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            lstm_par_results_ibm[f"{strat}_lstm_gat_par"] = {'test_metrics': pickle.load(f)}
        print(f"Skipping {strat} — already saved.")
        continue
    lstm_par_results_ibm[f"{strat}_lstm_gat_par"] = ibm_lstm_par.run_pipeline(df_ibm, cfg_ibm, strat)
    with open(save_path, 'wb') as f:
        pickle.dump(lstm_par_results_ibm[f"{strat}_lstm_gat_par"]['test_metrics'], f)
    print(f"Saved {strat} to Drive.")

print('\nIBM Parallel LSTM‖GAT done.')


############################################################
# LSTM‖GAT (parallel) | multi_relation
############################################################
  Fold 1/5
  [multi_relation] nodes=211,649 edges=916,245 deg(min/med/mean/max)=(1,5.0,4.3,9)
  [multi_relation] nodes=51,829 edges=267,895 deg(min/med/mean/max)=(1,5.0,5.2,9)
     TRAIN | F1 0.819 | P 0.822 | R 0.816 | AUC 0.982 | AP 0.884 | LL 0.2071 | Brier 0.0600
       VAL | F1 0.804 | P 0.815 | R 0.794 | AUC 0.977 | AP 0.868 | LL 0.2122 | Brier 0.0613
  Fold 2/5
     TRAIN | F1 0.820 | P 0.819 | R 0.822 | AUC 0.983 | AP 0.887 | LL 0.2112 | Brier 0.0610
       VAL | F1 0.785 | P 0.771 | R 0.799 | AUC 0.975 | AP 0.856 | LL 0.2648 | Brier 0.0779
  Fold 3/5
     TRAIN | F1 0.808 | P 0.800 | R 0.816 | AUC 0.980 | AP 0.878 | LL 0.2123 | Brier 0.0616
       VAL | F1 0.810 | P 0.814 | R 0.806 | AUC 0.980 | AP 0.879 | LL 0.2101 | Brier 0.0610
  Fold 4/5
     TRAIN | F1 0.823 | P 0.806 | R 0.840 | AUC 0.983 | AP 0.890 | LL 0.1982 

---
## Cell 7 — IBM: Homogeneous Account-Level GAT
~5–10 min (far fewer nodes than the transaction-level graphs).

In [8]:
import account_gat_homogeneous_model as ibm_acct_gat

acct_cfg_ibm = ibm_acct_gat.AccountFraudConfig()
acct_cfg_ibm.OUTCOME_DIR = '/content/outcomes/ibm'

df_accounts_ibm = ibm_acct_gat.build_account_dataframe(df_ibm, acct_cfg_ibm)
acct_results_ibm = ibm_acct_gat.run_all_strategies(df_accounts_ibm, acct_cfg_ibm)
print('\nIBM Homogeneous Account-Level GAT done.')

Aggregating transactions into account-level features...
  1,896 accounts | fraud rate 0.7083

############################################################
# Homogeneous Account GAT | similarity
############################################################
  Fold 1/5
  [similarity] accounts=1,213 edges=16,317 deg(min/med/mean/max)=(9,13.0,13.5,27)
  [similarity] accounts=304 edges=3,936 deg(min/med/mean/max)=(3,12.0,12.9,24)
  Early stop at epoch 80
     TRAIN | F1 0.907 | P 0.897 | R 0.917 | AUC 0.925 | AP 0.964 | LL 0.4698 | Brier 0.1446
       VAL | F1 0.904 | P 0.881 | R 0.928 | AUC 0.908 | AP 0.953 | LL 0.4831 | Brier 0.1507
  Fold 2/5
     TRAIN | F1 0.901 | P 0.888 | R 0.915 | AUC 0.918 | AP 0.961 | LL 0.4372 | Brier 0.1327
       VAL | F1 0.925 | P 0.907 | R 0.942 | AUC 0.938 | AP 0.966 | LL 0.4344 | Brier 0.1299
  Fold 3/5
     TRAIN | F1 0.923 | P 0.914 | R 0.932 | AUC 0.949 | AP 0.974 | LL 0.2852 | Brier 0.0884
       VAL | F1 0.929 | P 0.925 | R 0.933 | AUC 0.941 | AP 0.975 |

---
## Cell 8 — IBM: combined comparison table

In [9]:
import pandas as pd

all_results_ibm = {}
all_results_ibm.update(gatv2_results_ibm)
all_results_ibm.update(lstm_seq_results_ibm)
all_results_ibm.update(lstm_par_results_ibm)
all_results_ibm.update(acct_results_ibm)

rows = []
for key, res in all_results_ibm.items():
    m = res['test_metrics']
    rows.append({'run': key, 'model_arch': res.get('model_arch', ''),
                 'graph_strategy': res.get('graph_strategy', ''),
                 'f1': round(m['f1'], 4), 'prec': round(m['prec'], 4), 'rec': round(m['rec'], 4),
                 'auc': round(m['auc'], 4), 'ap': round(m['ap'], 4)})

df_summary_ibm = pd.DataFrame(rows).sort_values(['model_arch', 'f1'], ascending=[True, False])
os.makedirs('/content/outcomes/ibm', exist_ok=True)
df_summary_ibm.to_csv('/content/outcomes/ibm/all_architectures_summary.csv', index=False)
df_summary_ibm

,run,model_arch,graph_strategy,f1,prec,rec,auc,ap
9,similarity_account_gat,account_gat,similarity,0.9242,0.9066,0.9424,0.9121,0.9615
11,combined_account_gat,account_gat,combined,0.9112,0.8394,0.9964,0.8868,0.9449
10,shared_merchant_account_gat,account_gat,shared_merchant,0.9035,0.9151,0.8921,0.9073,0.9472
0,multi_relation_gatv2,gatv2,multi_relation,0.8279,0.8809,0.7808,0.9842,0.9080
1,hybrid_gatv2,gatv2,hybrid,0.8139,0.8787,0.7580,0.9839,0.9021
2,intra_group_gatv2,gatv2,intra_group,0.7700,0.7927,0.7486,0.9622,0.8372
6,multi_relation_lstm_gat_par,lstm_gat_par,multi_relation,0.8298,0.8614,0.8004,0.9842,0.9026
7,hybrid_lstm_gat_par,lstm_gat_par,hybrid,0.8109,0.8722,0.7576,0.9834,0.8987
8,intra_group_lstm_gat_par,lstm_gat_par,intra_group,0.7383,0.7073,0.7720,0.9593,0.8052
3,multi_relation_lstm_gat_seq,lstm_gat_seq,multi_relation,0.8926,0.9390,0.8505,0.9933,0.9579


---
# Part B — Sparkov Dataset

## Cell 9 — Load and preprocess
Set `SPARKOV_TRAIN_PATH` / `SPARKOV_TEST_PATH` to your actual
`fraudTrain.csv` / `fraudTest.csv` location before running.
Independent of Part A — safe to run this section on its own.

In [10]:
import os, sys
import sys

# ibm/ and sparkov/ both have modules with identical filenames
# (config.py, utils.py, gatv2_model.py, etc). Python caches imports by
# name in sys.modules, so re-importing after chdir silently reuses the
# IBM versions unless we evict them first.
for mod in ['config', 'utils', 'gatv2_model', 'lstm_gat_sequential_model',
            'lstm_gat_parallel_model', 'account_gat_homogeneous_model']:
    sys.modules.pop(mod, None)

import os, sys

SPARKOV_TRAIN_PATH = '/content/fraudTrain.csv'  # ← update if needed
SPARKOV_TEST_PATH  = '/content/fraudTest.csv'   # ← update if needed

os.chdir('/content/hybrid-gnn-lstm-fraud/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg_sparkov = sparkov_cfg.CardFraudConfig()
cfg_sparkov.OUTCOME_DIR = '/content/outcomes/sparkov'

import utils as sparkov_utils
df_sparkov = sparkov_utils.load_and_preprocess(
    train_path=SPARKOV_TRAIN_PATH, test_path=SPARKOV_TEST_PATH, cfg=cfg_sparkov)
print(f'{len(df_sparkov):,} transactions loaded.')

Combined: 1,852,394 transactions
After downsample: 52,394 rows (fraud rate: 0.1842)
Final shape: (52394, 27)
52,394 transactions loaded.


---
## Cell 10 — Sparkov: GATv2 baseline
~20–35 min on T4 GPU.

In [11]:
import gatv2_model as sparkov_gatv2

gatv2_results_sparkov = sparkov_gatv2.run_all_strategies(df_sparkov, cfg_sparkov)
print('\nSparkov GATv2 baseline done.')


############################################################
# GATv2 | multi_relation
############################################################
  Fold 1/5
  [multi_relation] nodes=33,815 edges=352,233 deg(min/med/mean/max)=(6,11.0,10.4,12)
  [multi_relation] nodes=8,094 edges=81,544 deg(min/med/mean/max)=(5,10.0,10.1,11)
     TRAIN | F1 0.962 | P 0.944 | R 0.980 | AUC 0.999 | AP 0.995 | LL 0.0542 | Brier 0.0151
       VAL | F1 0.928 | P 0.933 | R 0.923 | AUC 0.992 | AP 0.977 | LL 0.0933 | Brier 0.0258
  Fold 2/5
     TRAIN | F1 0.963 | P 0.974 | R 0.953 | AUC 0.999 | AP 0.994 | LL 0.0608 | Brier 0.0169
       VAL | F1 0.929 | P 0.947 | R 0.911 | AUC 0.992 | AP 0.976 | LL 0.1370 | Brier 0.0387
  Fold 3/5
     TRAIN | F1 0.962 | P 0.966 | R 0.957 | AUC 0.999 | AP 0.994 | LL 0.0635 | Brier 0.0175
       VAL | F1 0.917 | P 0.926 | R 0.909 | AUC 0.991 | AP 0.972 | LL 0.1730 | Brier 0.0513
  Fold 4/5
     TRAIN | F1 0.963 | P 0.972 | R 0.955 | AUC 0.999 | AP 0.995 | LL 0.0574 | Brier 0.0

---
## Cell 11 — Sparkov: Sequential LSTM→GAT
~30–50 min on T4 GPU.

In [12]:
import lstm_gat_sequential_model as sparkov_lstm_seq

lstm_seq_results_sparkov = sparkov_lstm_seq.run_all_strategies(df_sparkov, cfg_sparkov)
print('\nSparkov Sequential LSTM→GAT done.')


############################################################
# LSTM→GAT (sequential) | multi_relation
############################################################
  Fold 1/5
  [multi_relation] nodes=33,815 edges=352,233 deg(min/med/mean/max)=(6,11.0,10.4,12)
  [multi_relation] nodes=8,094 edges=81,544 deg(min/med/mean/max)=(5,10.0,10.1,11)
     TRAIN | F1 0.975 | P 0.992 | R 0.959 | AUC 1.000 | AP 0.998 | LL 0.0375 | Brier 0.0106
       VAL | F1 0.936 | P 0.962 | R 0.911 | AUC 0.994 | AP 0.982 | LL 0.1295 | Brier 0.0374
  Fold 2/5
     TRAIN | F1 0.976 | P 0.986 | R 0.967 | AUC 1.000 | AP 0.998 | LL 0.0352 | Brier 0.0099
       VAL | F1 0.942 | P 0.960 | R 0.925 | AUC 0.994 | AP 0.982 | LL 0.0959 | Brier 0.0270
  Fold 3/5
     TRAIN | F1 0.979 | P 0.983 | R 0.976 | AUC 0.999 | AP 0.998 | LL 0.0389 | Brier 0.0107
       VAL | F1 0.931 | P 0.946 | R 0.917 | AUC 0.993 | AP 0.978 | LL 0.1156 | Brier 0.0329
  Fold 4/5
     TRAIN | F1 0.977 | P 0.986 | R 0.969 | AUC 1.000 | AP 0.998 | LL 0.

---
## Cell 12 — Sparkov: Parallel LSTM‖GAT
~40–65 min on T4 GPU.

In [13]:
import lstm_gat_parallel_model as sparkov_lstm_par

lstm_par_results_sparkov = sparkov_lstm_par.run_all_strategies(df_sparkov, cfg_sparkov)
print('\nSparkov Parallel LSTM‖GAT done.')


############################################################
# LSTM‖GAT (parallel) | multi_relation
############################################################
  Fold 1/5
  [multi_relation] nodes=33,815 edges=352,233 deg(min/med/mean/max)=(6,11.0,10.4,12)
  [multi_relation] nodes=8,094 edges=81,544 deg(min/med/mean/max)=(5,10.0,10.1,11)
     TRAIN | F1 0.956 | P 0.940 | R 0.972 | AUC 0.998 | AP 0.993 | LL 0.0673 | Brier 0.0186
       VAL | F1 0.924 | P 0.943 | R 0.906 | AUC 0.992 | AP 0.977 | LL 0.0956 | Brier 0.0259
  Fold 2/5
     TRAIN | F1 0.963 | P 0.962 | R 0.965 | AUC 0.999 | AP 0.994 | LL 0.0584 | Brier 0.0165
       VAL | F1 0.921 | P 0.930 | R 0.912 | AUC 0.991 | AP 0.973 | LL 0.1211 | Brier 0.0345
  Fold 3/5
     TRAIN | F1 0.949 | P 0.986 | R 0.915 | AUC 0.999 | AP 0.994 | LL 0.0631 | Brier 0.0178
       VAL | F1 0.922 | P 0.960 | R 0.886 | AUC 0.993 | AP 0.976 | LL 0.1462 | Brier 0.0429
  Fold 4/5
     TRAIN | F1 0.957 | P 0.939 | R 0.976 | AUC 0.998 | AP 0.993 | LL 0.07

---
## Cell 13 — Sparkov: Homogeneous Account-Level GAT
~5–10 min.

In [14]:
import account_gat_homogeneous_model as sparkov_acct_gat

acct_cfg_sparkov = sparkov_acct_gat.CardAccountConfig()
acct_cfg_sparkov.OUTCOME_DIR = '/content/outcomes/sparkov'

df_accounts_sparkov = sparkov_acct_gat.build_account_dataframe(df_sparkov, acct_cfg_sparkov)
acct_results_sparkov = sparkov_acct_gat.run_all_strategies(df_accounts_sparkov, acct_cfg_sparkov)
print('\nSparkov Homogeneous Account-Level GAT done.')

Aggregating transactions into account-level features...
  999 accounts | fraud rate 0.9770

############################################################
# Homogeneous Account GAT | similarity
############################################################
  Fold 1/5
  [similarity] accounts=639 edges=8,669 deg(min/med/mean/max)=(9,13.0,13.6,29)
  [similarity] accounts=160 edges=2,164 deg(min/med/mean/max)=(9,13.0,13.5,26)
  Early stop at epoch 135
     TRAIN | F1 0.959 | P 1.000 | R 0.921 | AUC 0.992 | AP 1.000 | LL 0.2133 | Brier 0.0659
       VAL | F1 0.991 | P 0.981 | R 1.000 | AUC 0.930 | AP 0.999 | LL 0.0686 | Brier 0.0171
  Fold 2/5
     TRAIN | F1 0.961 | P 0.991 | R 0.933 | AUC 0.911 | AP 0.997 | LL 0.5048 | Brier 0.1713
       VAL | F1 0.994 | P 0.987 | R 1.000 | AUC 0.744 | AP 0.991 | LL 0.4390 | Brier 0.1395
  Fold 3/5
     TRAIN | F1 0.983 | P 0.985 | R 0.981 | AUC 0.926 | AP 0.998 | LL 0.4754 | Brier 0.1616
       VAL | F1 0.994 | P 0.988 | R 1.000 | AUC 0.924 | AP 0.999 | LL 

---
## Cell 14 — Sparkov: combined comparison table

In [15]:
all_results_sparkov = {}
all_results_sparkov.update(gatv2_results_sparkov)
all_results_sparkov.update(lstm_seq_results_sparkov)
all_results_sparkov.update(lstm_par_results_sparkov)
all_results_sparkov.update(acct_results_sparkov)

rows = []
for key, res in all_results_sparkov.items():
    m = res['test_metrics']
    rows.append({'run': key, 'model_arch': res.get('model_arch', ''),
                 'graph_strategy': res.get('graph_strategy', ''),
                 'f1': round(m['f1'], 4), 'prec': round(m['prec'], 4), 'rec': round(m['rec'], 4),
                 'auc': round(m['auc'], 4), 'ap': round(m['ap'], 4)})

df_summary_sparkov = pd.DataFrame(rows).sort_values(['model_arch', 'f1'], ascending=[True, False])
os.makedirs('/content/outcomes/sparkov', exist_ok=True)
df_summary_sparkov.to_csv('/content/outcomes/sparkov/all_architectures_summary.csv', index=False)
df_summary_sparkov

,run,model_arch,graph_strategy,f1,prec,rec,auc,ap
9,similarity_account_gat,account_gat,similarity,0.9924,0.9850,1.0000,0.9814,0.9997
10,shared_merchant_account_gat,account_gat,shared_merchant,0.9924,0.9850,1.0000,0.9983,1.0000
11,combined_account_gat,account_gat,combined,0.9924,0.9850,1.0000,0.9966,0.9999
2,intra_group_gatv2,gatv2,intra_group,0.9098,0.9431,0.8788,0.9918,0.9714
0,multi_relation_gatv2,gatv2,multi_relation,0.9088,0.9569,0.8653,0.9943,0.9782
1,hybrid_gatv2,gatv2,hybrid,0.8623,0.9124,0.8175,0.9826,0.9418
6,multi_relation_lstm_gat_par,lstm_gat_par,multi_relation,0.9172,0.9424,0.8934,0.9940,0.9776
8,intra_group_lstm_gat_par,lstm_gat_par,intra_group,0.9135,0.9357,0.8924,0.9921,0.9723
7,hybrid_lstm_gat_par,lstm_gat_par,hybrid,0.8694,0.9270,0.8185,0.9835,0.9448
5,intra_group_lstm_gat_seq,lstm_gat_seq,intra_group,0.9438,0.9582,0.9298,0.9972,0.9886


---
## Cell 15 — Download everything

In [16]:
!cd /content/outcomes && zip -qr /content/outcomes.zip .
from google.colab import files
files.download('/content/outcomes.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
!grep -n "GATv2Conv\|GATConv" /content/hybrid-gnn-lstm-fraud/ibm/lstm_gat_parallel_model.py
!grep -n "lstm_hidden = lstm_hidden or" /content/hybrid-gnn-lstm-fraud/sparkov/lstm_gat_sequential_model.py
!grep -n "lstm_hidden = lstm_hidden or" /content/hybrid-gnn-lstm-fraud/sparkov/lstm_gat_parallel_model.py

30:from torch_geometric.nn import GATv2Conv
153:        self.gat1 = GATv2Conv(in_dim, hidden, heads=heads, dropout=dropout)
154:        self.gat2 = GATv2Conv(hidden*heads, hidden, heads=heads, dropout=dropout)
112:        lstm_hidden = lstm_hidden or 64
120:        lstm_hidden = lstm_hidden or 64
